# DiDAQt Controller Scalability Benchmark

Measures controller preprocessing time and failover decision time across
varying topology sizes (path counts) and path lengths (switch counts).

Creates a single FABRIC VM for benchmarking, uploads the DiDAQt source,
generates topologies, and runs the benchmark program.

In [ ]:
# ---- Configuration (all constants here for notebook resume) ----

SLICE_NAME = "didaqt_benchmark"
NODE_NAME  = "bench"
CORES      = 8
RAM        = 32
DISK       = 100
IMAGE      = "default_ubuntu_22"

# Benchmark parameters
PATH_COUNTS    = []
i = 16
while (i * 6 < 100000):
    PATH_COUNTS.append(i*6)
    i *= 2
print(PATH_COUNTS)
SWITCH_COUNTS  = [1, 2, 3, 4, 5]

# Remote paths
REMOTE_DIR    = "/home/ubuntu/didaqt"
TOPO_DIR      = f"{REMOTE_DIR}/topologies"
RESULTS_DIR   = f"{REMOTE_DIR}/results"
BENCH_BIN     = f"{REMOTE_DIR}/build/bench_ctrl"
GEN_SCRIPT    = f"{REMOTE_DIR}/benchmarking/gen_topology.py"

# Local paths
LOCAL_REPO    = ".."  # relative to artifact/
LOCAL_RESULTS = "benchmark_results"

In [ ]:
# ---- Create FABRIC slice ----

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()

try:
    slice = fablib.get_slice(name=SLICE_NAME)
    print(f"Slice '{SLICE_NAME}' already exists, reusing.")
except:
    slice = fablib.new_slice(name=SLICE_NAME)
    node = slice.add_node(
        name=NODE_NAME,
        cores=CORES,
        ram=RAM,
        disk=DISK,
        image=IMAGE,
    )
    slice.submit()
    print(f"Slice '{SLICE_NAME}' submitted.")

slice.wait_ssh(progress=True)
node = slice.get_node(name=NODE_NAME)
print(f"Node ready: {node.get_management_ip()}")

In [ ]:
# ---- Install dependencies and upload source ----

node = slice.get_node(name=NODE_NAME)

# Install build deps
stdout, stderr = node.execute(
    "sudo apt-get update -qq && "
    "sudo apt-get install -y -qq gcc make libyaml-dev python3",
    quiet=True,
)
print("Dependencies installed.")

# Upload repo
import os, subprocess, tempfile

repo_root = os.path.abspath(LOCAL_REPO)
tarball = os.path.join(tempfile.gettempdir(), "didaqt_src.tar.gz")
subprocess.run(
    ["tar", "czf", tarball,
     "--exclude=.git", "--exclude=build", "--exclude=working",
     "--exclude=artifact/didaqt_experiment.ipynb",
     "--exclude=artifact/didaqt_benchmark.ipynb",
     "-C", os.path.dirname(repo_root),
     os.path.basename(repo_root)],
    check=True,
)
node.upload_file(tarball, "/tmp/didaqt_src.tar.gz")
node.execute(
    f"rm -rf {REMOTE_DIR} && "
    f"tar xzf /tmp/didaqt_src.tar.gz -C /home/ubuntu"
)
print("Source uploaded.")

In [ ]:
# ---- Compile ----

node = slice.get_node(name=NODE_NAME)

stdout, stderr = node.execute(
    f"cd {REMOTE_DIR} && make clean && make && make bench"
)
print(stdout)
if stderr.strip():
    print("STDERR:", stderr)

In [ ]:
# ---- Run benchmarks ----

import csv
from concurrent.futures import ThreadPoolExecutor, as_completed

node = slice.get_node(name=NODE_NAME)

# Create directories on remote
node.execute(f"mkdir -p {TOPO_DIR} {RESULTS_DIR}")

preprocess_rows = []  # (active_receivers, switch_count, preprocess_ns)
decision_rows = []    # (active_receivers, switch_count, decision_ns)

for target in PATH_COUNTS:
    for sc in SWITCH_COUNTS:
        label = f"p{target}_s{sc}"
        topo_file = f"{TOPO_DIR}/{label}.yaml"
        print(f"\n--- {label}: generating topology ---")

        # Generate topology (target = active receivers)
        stdout, stderr = node.execute(
            f"python3 {GEN_SCRIPT} {target} {sc} {topo_file}"
        )
        if stderr.strip():
            print(f"  gen stderr: {stderr.strip()}")

        # Show topology size
        stdout, _ = node.execute(f"wc -l {topo_file} | awk '{{print $1}}'")
        print(f"  YAML lines: {stdout.strip()}")

        # Run benchmark
        print(f"  running benchmark...")
        stdout, stderr = node.execute(
            f"{BENCH_BIN} {topo_file} {sc}",
            quiet=True,
        )
        if stderr.strip():
            print(f"  bench stderr: {stderr.strip()}")

        # Parse output
        out_lines = stdout.strip().split("\n")
        if len(out_lines) < 11:
            print(f"  ERROR: unexpected output ({len(out_lines)} lines)")
            print(stdout)
            continue

        # Line 1: active_receivers,switch_count,preprocess_ns
        parts = out_lines[0].split(",")
        active_recv = int(parts[0])
        preprocess_ns = int(parts[2])
        preprocess_rows.append((active_recv, sc, preprocess_ns))
        print(f"  active_receivers={active_recv} preprocess={preprocess_ns/1e6:.3f}ms")

        # Lines 2-11: decision_ns (skip FAIL lines)
        trial_times = []
        for i in range(1, 11):
            line = out_lines[i].strip()
            if line == "FAIL":
                print(f"  trial {i-1}: FAIL")
                continue
            dns = int(line)
            decision_rows.append((active_recv, sc, dns))
            trial_times.append(dns)

        if trial_times:
            avg_us = sum(trial_times) / len(trial_times) / 1000
            print(f"  decision avg={avg_us:.1f}us "
                  f"min={min(trial_times)/1000:.1f}us "
                  f"max={max(trial_times)/1000:.1f}us")
        else:
            print(f"  all trials FAILED")

        # Clean up large topology file to save disk
        node.execute(f"rm -f {topo_file}")

print("\n=== All benchmarks complete ===")

In [ ]:
# ---- Save results to CSV ----

import os

os.makedirs(LOCAL_RESULTS, exist_ok=True)

preprocess_csv = os.path.join(LOCAL_RESULTS, "preprocess_times.csv")
with open(preprocess_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["active_receivers", "switches_per_path", "preprocess_ns"])
    for row in preprocess_rows:
        w.writerow(row)
print(f"Wrote {preprocess_csv} ({len(preprocess_rows)} rows)")

decision_csv = os.path.join(LOCAL_RESULTS, "decision_times.csv")
with open(decision_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["active_receivers", "switches_per_path", "decision_ns"])
    for row in decision_rows:
        w.writerow(row)
print(f"Wrote {decision_csv} ({len(decision_rows)} rows)")

In [ ]:
# ---- Summary table ----

print(f"{'Recv':>8}  {'SW':>3}  {'Preprocess (ms)':>16}  "
      f"{'Decision avg (us)':>18}  {'min':>8}  {'max':>8}")
print("-" * 75)

for target in PATH_COUNTS:
    for sc in SWITCH_COUNTS:
        pp = [r for r in preprocess_rows if r[1] == sc
              and any(d[0] == r[0] and d[1] == sc for d in decision_rows)]
        if not pp:
            continue
        ar = pp[0][0]
        pp_ns = pp[0][2]

        dts = [r[2] for r in decision_rows
               if r[0] == ar and r[1] == sc]
        if not dts:
            continue

        avg_us = sum(dts) / len(dts) / 1000
        min_us = min(dts) / 1000
        max_us = max(dts) / 1000

        print(f"{ar:>8}  {sc:>3}  {pp_ns/1e6:>16.3f}  "
              f"{avg_us:>18.1f}  {min_us:>8.1f}  {max_us:>8.1f}")

In [ ]:
# ---- Plot results ----

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pp_df = pd.read_csv(os.path.join(LOCAL_RESULTS, "preprocess_times.csv"))
dt_df = pd.read_csv(os.path.join(LOCAL_RESULTS, "decision_times.csv"))

# Aggregate decision times: median per (active_receivers, switches_per_path)
dt_agg = dt_df.groupby(["active_receivers", "switches_per_path"])["decision_ns"].median().reset_index()

STYLES = {
    1: {"color": "#377eb8", "marker": "o", "linestyle": "-"},
    2: {"color": "#ff7f00", "marker": "s", "linestyle": "--"},
    3: {"color": "#4daf4a", "marker": "^", "linestyle": "-."},
    4: {"color": "#f781bf", "marker": "X", "linestyle": ":"},
    5: {"color": "#a65628", "marker": "p", "linestyle": "-"},
}

# IEEE dual-column: column width ~3.5 in, use ~3.4 in with ~2.4 in height
COL_WIDTH = 3.4
COL_HEIGHT = 2.4

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "lines.markersize": 4,
    "lines.linewidth": 1,
})

MS = 1e6  # ns -> ms
US = 1e3  # ns -> us

def stage_label(sc):
    return f"{sc} stage{'s' if sc > 1 else ''}"

# ---- Preprocessing time (log-log) ----
fig_pp, ax_pp = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for sc in sorted(pp_df["switches_per_path"].unique()):
    sub = pp_df[pp_df["switches_per_path"] == sc].sort_values("active_receivers")
    ax_pp.plot(sub["active_receivers"], sub["preprocess_ns"] / MS,
               label=stage_label(sc), **STYLES[sc])
ax_pp.set_xscale("log")
ax_pp.set_yscale("log")
ax_pp.set_xlabel("Number of active receivers")
ax_pp.set_ylabel("Preprocessing time (ms)")
ax_pp.legend()
ax_pp.grid(True, which="both", ls=":", lw=0.5)
fig_pp.tight_layout()
fig_pp.savefig(os.path.join(LOCAL_RESULTS, "preprocess_time.pdf"), bbox_inches="tight")
plt.show()

# ---- Decision time ----
fig_dt, ax_dt = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for sc in sorted(dt_agg["switches_per_path"].unique()):
    sub = dt_agg[dt_agg["switches_per_path"] == sc].sort_values("active_receivers")
    ax_dt.plot(sub["active_receivers"], sub["decision_ns"] / US,
               label=stage_label(sc), **STYLES[sc])
ax_dt.set_xscale("log")
ax_dt.set_yscale("log")
ax_dt.set_xlabel("Number of active receivers")
ax_dt.set_ylabel("Decision time (\u00b5s)")
ax_dt.legend()
ax_dt.grid(True, which="both", ls=":", lw=0.5)
fig_dt.tight_layout()
fig_dt.savefig(os.path.join(LOCAL_RESULTS, "decision_time.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# ---- Run preprocessing & worst-case benchmarks ----

import csv

node = slice.get_node(name=NODE_NAME)
BENCH_EXTRA = f"{REMOTE_DIR}/build/bench_ctrl_extra"

node.execute(f"mkdir -p {TOPO_DIR}")

preprocess2_rows = []  # (active_receivers, switch_count, preprocess_ns)
worstcase_rows = []    # (active_receivers, switch_count, worstcase_ns)

for target in PATH_COUNTS:
    for sc in SWITCH_COUNTS:
        label = f"p{target}_s{sc}"
        topo_file = f"{TOPO_DIR}/{label}.yaml"
        print(f"\n--- {label}: generating topology ---")

        stdout, stderr = node.execute(
            f"python3 {GEN_SCRIPT} {target} {sc} {topo_file}"
        )
        if stderr.strip():
            print(f"  gen stderr: {stderr.strip()}")

        print(f"  running extra benchmark...")
        stdout, stderr = node.execute(
            f"{BENCH_EXTRA} {topo_file} {sc}",
            quiet=True,
        )
        if stderr.strip():
            print(f"  bench stderr: {stderr.strip()}")

        out_lines = stdout.strip().split("\n")
        if len(out_lines) < 11:
            print(f"  ERROR: unexpected output ({len(out_lines)} lines)")
            print(stdout)
            node.execute(f"rm -f {topo_file}")
            continue

        # Line 1: active_receivers,switch_count,preprocess_ns,paths_per_sender
        parts = out_lines[0].split(",")
        active_recv = int(parts[0])
        pp_ns = int(parts[2])
        paths_per_sender = int(parts[3])
        preprocess2_rows.append((active_recv, sc, pp_ns))
        print(f"  active_receivers={active_recv} preprocess={pp_ns/1e6:.3f}ms "
              f"paths_per_sender={paths_per_sender}")

        # Lines 2-11: worst_case_decision_ns
        trial_times = []
        for i in range(1, 11):
            line = out_lines[i].strip()
            if line == "FAIL":
                print(f"  trial {i-1}: FAIL")
                continue
            wns = int(line)
            worstcase_rows.append((active_recv, sc, wns))
            trial_times.append(wns)

        if trial_times:
            avg_us = sum(trial_times) / len(trial_times) / 1000
            print(f"  worst-case avg={avg_us:.1f}us "
                  f"min={min(trial_times)/1000:.1f}us "
                  f"max={max(trial_times)/1000:.1f}us")

        node.execute(f"rm -f {topo_file}")

# ---- Save extra CSVs ----
preprocess2_csv = os.path.join(LOCAL_RESULTS, "preprocess_times_extra.csv")
with open(preprocess2_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["active_receivers", "switches_per_path", "preprocess_ns"])
    for row in preprocess2_rows:
        w.writerow(row)
print(f"\nWrote {preprocess2_csv} ({len(preprocess2_rows)} rows)")

worstcase_csv = os.path.join(LOCAL_RESULTS, "worstcase_times.csv")
with open(worstcase_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["active_receivers", "switches_per_path", "worstcase_ns"])
    for row in worstcase_rows:
        w.writerow(row)
print(f"Wrote {worstcase_csv} ({len(worstcase_rows)} rows)")

print("\n=== Extra benchmarks complete ===")

In [ ]:
# ---- Plot preprocessing & worst-case results ----

import pandas as pd
import matplotlib.pyplot as plt
import os

pp2_df = pd.read_csv(os.path.join(LOCAL_RESULTS, "preprocess_times_extra.csv"))
wc_df = pd.read_csv(os.path.join(LOCAL_RESULTS, "worstcase_times.csv"))

# Aggregate worst-case: median per (active_receivers, switches_per_path)
wc_agg = wc_df.groupby(
    ["active_receivers", "switches_per_path"])["worstcase_ns"].median().reset_index()

STYLES = {
    1: {"color": "#377eb8", "marker": "o", "linestyle": "-"},
    2: {"color": "#ff7f00", "marker": "s", "linestyle": "--"},
    3: {"color": "#4daf4a", "marker": "^", "linestyle": "-."},
    4: {"color": "#f781bf", "marker": "X", "linestyle": ":"},
    5: {"color": "#a65628", "marker": "p", "linestyle": "-"},
}

COL_WIDTH = 3.4
COL_HEIGHT = 2.4

plt.rcParams.update({
    "font.size": 8,
    "axes.labelsize": 8,
    "legend.fontsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "lines.markersize": 4,
    "lines.linewidth": 1,
})

MS = 1e6  # ns -> ms
US = 1e3  # ns -> us

def stage_label(sc):
    return f"{sc} stage{'s' if sc > 1 else ''}"

# ---- Preprocessing time (log-log) ----
fig_pp, ax_pp = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for sc in sorted(pp2_df["switches_per_path"].unique()):
    sub = pp2_df[pp2_df["switches_per_path"] == sc].sort_values("active_receivers")
    ax_pp.plot(sub["active_receivers"], sub["preprocess_ns"] / MS,
               label=stage_label(sc), **STYLES[sc])
ax_pp.set_xscale("log")
ax_pp.set_yscale("log")
ax_pp.set_xlabel("Number of active receivers")
ax_pp.set_ylabel("Preprocessing time (ms)")
ax_pp.legend()
ax_pp.grid(True, which="both", ls=":", lw=0.5)
fig_pp.tight_layout()
fig_pp.savefig(os.path.join(LOCAL_RESULTS, "preprocess_time_extra.pdf"), bbox_inches="tight")
plt.show()

# ---- Worst-case decision time ----
fig_wc, ax_wc = plt.subplots(figsize=(COL_WIDTH, COL_HEIGHT))
for sc in sorted(wc_agg["switches_per_path"].unique()):
    sub = wc_agg[wc_agg["switches_per_path"] == sc].sort_values("active_receivers")
    ax_wc.plot(sub["active_receivers"], sub["worstcase_ns"] / US,
               label=stage_label(sc), **STYLES[sc])
ax_wc.set_xscale("log")
ax_wc.set_yscale("log")
ax_wc.set_xlabel("Number of active receivers")
ax_wc.set_ylabel("Worst-case decision time (\u00b5s)")
ax_wc.legend()
ax_wc.grid(True, which="both", ls=":", lw=0.5)
fig_wc.tight_layout()
fig_wc.savefig(os.path.join(LOCAL_RESULTS, "worstcase_time.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# ---- Cleanup (optional) ----

# slice.delete()